# 02 Symbol Optimize

Notebook n?y d?ng ?? t?i ?u **m?t symbol** cho Combo. Workflow:
- ch?y grid search nhanh b?ng `backtest_fast()`
- xem top candidates
- ch?n candidate t?t nh?t ?? validate b?ng full backtest


In [ ]:
# Bootstrap: add repo root + core_python to sys.path
import sys
from pathlib import Path

def _find_root(start: Path, marker: str = 'config.py') -> Path:
    for p in [start, *start.parents]:
        if (p / marker).exists():
            return p
    raise RuntimeError(f'Could not locate repo root containing {marker!r}')

ROOT = _find_root(Path.cwd())
CORE = ROOT / 'core_python'
for p in (str(ROOT), str(CORE)):
    if p not in sys.path:
        sys.path.insert(0, p)

print('ROOT =', ROOT)
print('CORE =', CORE)


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

plt.style.use('dark_background')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

from strategies.combo.config import SYMBOLS, get_symbol_search_space, summary as strategy_summary
from strategies.combo.symbol.backtest import run_symbol_backtest
from strategies.combo.symbol.optimize import run_symbol_grid_search

print(strategy_summary())
print('Symbols:', ', '.join(SYMBOLS.keys()))


In [ ]:
SYMBOL = 'US30'
ACCOUNT_MODE = 'standard'
DATE_FROM = '2022-01-01'
DATE_TO = None
INITIAL_BALANCE = 100_000.0
MAX_BARS = 50000
TOP_N = 15

search_space = get_symbol_search_space(SYMBOL)
search_space


In [ ]:
grid = run_symbol_grid_search(
    SYMBOL,
    date_from=DATE_FROM,
    date_to=DATE_TO,
    init_eq=INITIAL_BALANCE,
    account_mode=ACCOUNT_MODE,
    max_bars=MAX_BARS,
    search_space=search_space,
)

print('Candidates =', len(grid))
display(grid.head(TOP_N))


In [ ]:
if grid.empty:
    raise RuntimeError('No candidates returned from grid search.')

best = grid.iloc[0]
best_params = {
    'ktp': float(best['ktp']),
    'x': float(best['x']),
    'ma_period': int(best['ma_period']),
    'trailing_activation': float(best['trailing_activation']),
}

best_params


In [ ]:
validation = run_symbol_backtest(
    SYMBOL,
    init_eq=INITIAL_BALANCE,
    account_mode=ACCOUNT_MODE,
    date_from=DATE_FROM,
    date_to=DATE_TO,
    max_bars=MAX_BARS,
    symbol_overrides=best_params,
)

validation_metrics = pd.Series({k: v for k, v in validation.metrics.items() if k != 'monthly_pnl_table'})
display(validation_metrics.to_frame('value'))


In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
validation.equity.plot(ax=ax, color='#FFD93D', lw=1.8, title=f'Validated equity: {SYMBOL}')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
compare_cols = [c for c in ['ktp','x','trailing_activation','ma_period','sharpe','profit_factor','total_return','max_drawdown','win_rate','score'] if c in grid.columns]
display(grid[compare_cols].head(TOP_N))
